# Quantum Lock: a small quantum enigma machine

Phase-modulation quantum data locking (Lupo, Wilde & Lloyd, *PRA* **90**, 022326, 2014).

- **Alice** encodes message $m$ with key $k$ as $|\psi_{m,k}\rangle = D_k F |m\rangle$ ($F$ = QFT, $D_k$ = random diagonal phase mask)
- **Bob** decodes with $F^\dagger D_k^\dagger$ and measures in the computational basis
- We compare: **correct key**, **wrong key**, **no key** (no phase correction), **direct measurement**

Install once: `pip install "qiskit>=1.2" matplotlib pylatexenc` (cell below).

Running all cells writes the files used by the report:
`results.tex`, `table_n.tex`, `table_K.tex`, `fig_circuit.pdf`, `fig_distributions.pdf`

In [ ]:
%pip install -q "qiskit>=1.2" matplotlib pylatexenc

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Gate
from qiskit.circuit.library import QFTGate, DiagonalGate
from qiskit.quantum_info import Statevector, Operator

SEED = 2026            # fixed seed -> the same phase codebook every run
N_QUBITS = 6           # message qubits n, d = 2**n = 64 messages
N_KEYS = 4             # K = 4 keys (2-bit classical key)
M_EX, M_EX2 = 37, 12   # example messages for the figures
K_EX, KP_EX = 0, 2     # true key and wrong key used in the examples

## 1. Phase codebook

For each key $k$: $d$ angles $\theta_{k,x}$, uniform in $[0, 2\pi)$ (or in $\{0, \pi\}$ for the binary variant).

In [ ]:
def make_codebook(n, K, seed=SEED, binary=False):
    """Phase angles theta[k, x] for K keys and d = 2**n basis states."""
    rng = np.random.default_rng(seed)
    d = 2**n
    if binary:
        return np.pi * rng.integers(0, 2, size=(K, d))
    return rng.uniform(0, 2 * np.pi, size=(K, d))

theta = make_codebook(N_QUBITS, N_KEYS)
print("codebook shape (K, d):", theta.shape)

## 2. Circuits

Only the message register is quantum; the key is a classical index choosing the phase circuit.

In [ ]:
def prepare_message(n, m):
    """Basis state |m> (Qiskit convention: qubit q holds bit q of m)."""
    qc = QuantumCircuit(n)
    for q in range(n):
        if (m >> q) & 1:
            qc.x(q)
    return qc

def lock_circuit(n, theta_k):
    """Alice: |m> -> D_k F |m>."""
    qc = QuantumCircuit(n)
    qc.append(QFTGate(n), range(n))
    qc.append(DiagonalGate(list(np.exp(1j * theta_k))), range(n))
    return qc

def unlock_circuit(n, theta_k=None):
    """Receiver: D_k^dagger then F^dagger.  theta_k=None -> no phase correction (F^dagger only)."""
    qc = QuantumCircuit(n)
    if theta_k is not None:
        qc.append(DiagonalGate(list(np.exp(-1j * theta_k))), range(n))
    qc.append(QFTGate(n).inverse(), range(n))
    return qc

def probs(sv):
    """Outcome distribution as an array indexed by the integer outcome j."""
    p = np.zeros(2**sv.num_qubits)
    for bits, val in sv.probabilities_dict().items():
        p[int(bits, 2)] = val
    return p

In [ ]:
# Sanity check on one message
sv = Statevector.from_instruction(prepare_message(N_QUBITS, M_EX).compose(lock_circuit(N_QUBITS, theta[K_EX])))
print("|amplitudes| of locked state:", np.unique(np.round(np.abs(sv.data), 8)))   # all 1/sqrt(64) = 0.125
print("P(m) with correct key:", round(probs(sv.evolve(unlock_circuit(N_QUBITS, theta[K_EX])))[M_EX], 6))
print("P(m) with wrong key  :", round(probs(sv.evolve(unlock_circuit(N_QUBITS, theta[KP_EX])))[M_EX], 6))

## 3. Worked example: one qubit ($d = 2$)

$F = H$ and $D_k = \mathrm{diag}(e^{i a_k}, e^{i b_k})$. A wrong key leaves a relative phase $\Delta$ and

$$P(\text{recover } m) = \frac{|1 + e^{i\Delta}|^2}{4} = \cos^2(\Delta/2).$$

We check the formula against the circuit.

In [ ]:
th1 = make_codebook(1, 2)                                               # 2 keys on one qubit
we_delta = np.mod((th1[0, 1] - th1[0, 0]) - (th1[1, 1] - th1[1, 0]), 2 * np.pi)
sv1 = Statevector.from_instruction(prepare_message(1, 0).compose(lock_circuit(1, th1[0])))
we_pcirc = probs(sv1.evolve(unlock_circuit(1, th1[1])))[0]
we_pform = np.cos(we_delta / 2) ** 2
print(f"Delta = {we_delta:.3f} rad | circuit P(m) = {we_pcirc:.6f} | cos^2(Delta/2) = {we_pform:.6f}")

In [ ]:
# Circuit figure (drawing only): compact labelled blocks instead of the 64-parameter DiagonalGate
def block(name, label):
    return Gate(name=name, num_qubits=N_QUBITS, params=[], label=label)

qc = prepare_message(N_QUBITS, M_EX)                 # X gates prepare |37> = |100101>
qc.barrier(label="Alice: lock")
qc.append(block("qft_blk", "QFT"), range(N_QUBITS))
qc.append(block("dk_blk", "$D_k$"), range(N_QUBITS))
qc.barrier(label="Bob: unlock")
qc.append(block("dkp_blk", "$D_{k'}^\\dagger$"), range(N_QUBITS))
qc.append(block("iqft_blk", "QFT$^\\dagger$"), range(N_QUBITS))
qc.measure_all()
fig = qc.draw("mpl", fold=-1)
fig.savefig("fig_circuit.pdf", bbox_inches="tight")
fig

## 4. Simulate all message-key pairs

Strategy $s$: $0..K-1$ = decode with key $s$, $K$ = no phase correction, $K+1$ = direct measurement.

`P[i, k, s, j]` = probability of outcome $j$ for message `messages[i]` locked with key $k$.

In [ ]:
def run_experiment(n, K, seed=SEED, binary=False, n_messages=None):
    d = 2**n
    theta = make_codebook(n, K, seed, binary)
    if n_messages is None or n_messages >= d:
        messages = np.arange(d)                                     # all messages (n <= 6)
    else:                                                           # fixed random subset (larger n)
        messages = np.sort(np.random.default_rng(seed + 1).choice(d, n_messages, replace=False))

    locks = [lock_circuit(n, theta[k]) for k in range(K)]
    # decoders depend only on the key: compile them once into Operators
    unlocks = [Operator(unlock_circuit(n, theta[s])) for s in range(K)] + [Operator(unlock_circuit(n))]

    P = np.zeros((len(messages), K, K + 2, d))
    psi = np.zeros((len(messages), K, d), dtype=complex)
    for i, m in enumerate(messages):
        for k in range(K):
            sv = Statevector.from_instruction(prepare_message(n, m).compose(locks[k]))   # |psi_{m,k}>
            psi[i, k] = sv.data
            for s, U in enumerate(unlocks):
                P[i, k, s] = probs(sv.evolve(U))
            P[i, k, K + 1] = probs(sv)                               # direct measurement
    return {"n": n, "d": d, "K": K, "theta": theta, "messages": messages, "P": P, "psi": psi}

## 5. Metrics: recovery probability, entropy, Eve's information

In [ ]:
def entropy_bits(p):
    p = np.asarray(p)
    p = p[p > 1e-12]
    return float(-np.sum(p * np.log2(p)))

def haar_entropy(d):
    """Average outcome entropy (bits) of a random pure state in dimension d: (H_d - 1)/ln 2."""
    return float(np.sum(1.0 / np.arange(2, d + 1)) / np.log(2))

def mutual_info(channel):
    """I(M;J) in bits for uniform messages; channel[i, j] = P(j | message i)."""
    return entropy_bits(channel.mean(axis=0)) - np.mean([entropy_bits(row) for row in channel])

def holevo_bits(psi):
    """Holevo quantity of Eve's ensemble rho_m = (1/K) sum_k |psi_mk><psi_mk| (uniform messages)."""
    rhos = [np.einsum("ka,kb->ab", p, p.conj()) / len(p) for p in psi]
    vn = lambda rho: entropy_bits(np.clip(np.linalg.eigvalsh(rho), 0, None))
    return vn(np.mean(rhos, axis=0)) - np.mean([vn(r) for r in rhos])

def summarize(res):
    P, msgs, K, d = res["P"], res["messages"], res["K"], res["d"]
    rec = np.array([P[i, :, :, m] for i, m in enumerate(msgs)])   # rec[i,k,s] = P(recover m)
    H = np.apply_along_axis(entropy_bits, -1, P)                  # H[i,k,s] = entropy of outcomes
    same = np.eye(K, dtype=bool)                                  # decoding key == true key
    eve = [mutual_info(P[:, :, s, :].mean(axis=1)) for s in range(K + 2)]   # Eve: k unknown
    return {
        "n": res["n"], "d": d, "K": K,
        "p_correct": rec[:, :, :K][:, same].mean(),
        "dev_correct": np.abs(1 - rec[:, :, :K][:, same]).max(),
        "H_correct": H[:, :, :K][:, same].mean(),
        "p_wrong": rec[:, :, :K][:, ~same].mean(),
        "p_wrong_pairs": rec[:, :, :K][:, ~same].mean(axis=0),
        "H_wrong": H[:, :, :K][:, ~same].mean(),
        "p_nocorr": rec[:, :, K].mean(), "H_nocorr": H[:, :, K].mean(),
        "p_direct": rec[:, :, K + 1].mean(), "H_direct": H[:, :, K + 1].mean(),
        "I_guess": max(eve[:K]), "I_nocorr": eve[K], "I_direct": eve[K + 1],
        "chi": holevo_bits(res["psi"]),
    }

def show(S):
    print(f"n={S['n']} d={S['d']} K={S['K']}  (1/d={1/S['d']:.4f}, random-state entropy={haar_entropy(S['d']):.2f} bits)")
    print(f"  correct key         P(m)={S['p_correct']:.4f}  H={S['H_correct']:.2f}  max|1-P|={S['dev_correct']:.1e}")
    print(f"  wrong key           P(m)={S['p_wrong']:.4f}  H={S['H_wrong']:.2f}")
    print(f"  no phase correction P(m)={S['p_nocorr']:.4f}  H={S['H_nocorr']:.2f}")
    print(f"  direct measurement  P(m)={S['p_direct']:.4f}  H={S['H_direct']:.2f}")
    print(f"  Eve: key guess I={S['I_guess']:.2f}, no correction I={S['I_nocorr']:.2f}, direct I={abs(S['I_direct']):.2f} bits")
    print(f"  Holevo chi={S['chi']:.2f} bits  (n - log2 K = {np.log2(S['d'] / S['K']):.2f})")

## 6. Main experiment: $n = 6$, $d = M = 64$, $K = 4$ (256 message-key pairs)

In [ ]:
main = run_experiment(N_QUBITS, N_KEYS)
S = summarize(main)
show(S)

rec_pairs = np.array([main["P"][i, :, :N_KEYS, m] for i, m in enumerate(main["messages"])]).mean(axis=0)
print("\nP(recover m): rows = true key k, columns = decoding key k'")
print(np.round(rec_pairs, 4))

In [ ]:
# Check: every distribution depends only on j - m, i.e. P(.|m) = P(.|0) shifted by m
P = main["P"]
shift_dev = max(np.abs(P[m, k, s] - np.roll(P[0, k, s], m)).max()
                for m in main["messages"] for k in range(N_KEYS) for s in range(N_KEYS + 2))
print(f"max deviation from the shift structure: {shift_dev:.1e}")

In [ ]:
# Distributions for two selected messages (rows) and the four decoding strategies (columns)
d = 2**N_QUBITS
cols = [(K_EX, "correct key"), (KP_EX, f"wrong key k'={KP_EX}"),
        (N_KEYS, "no key (QFT$^\\dagger$ only)"), (N_KEYS + 1, "direct measurement")]
fig, axes = plt.subplots(2, 4, figsize=(13, 4.6), sharex=True)
for row, m in enumerate([M_EX, M_EX2]):
    for col, (s, title) in enumerate(cols):
        ax = axes[row, col]
        p = main["P"][m, K_EX, s]
        ax.bar(range(d), p, width=1.0, color=["tab:red" if j == m else "tab:blue" for j in range(d)])
        ax.set_title(f"m={m}, {title}\nP(m)={p[m]:.3f}, H={entropy_bits(p):.2f} bits", fontsize=8)
        ax.tick_params(labelsize=7)
    axes[row, 0].set_ylabel("probability", fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("outcome j", fontsize=8)
fig.tight_layout()
fig.savefig("fig_distributions.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Simpler variant: phases restricted to {0, pi}
main_bin = run_experiment(N_QUBITS, N_KEYS, binary=True)
S_bin = summarize(main_bin)
show(S_bin)

## 7. Codebook-to-codebook variation (error bars)

One codebook gives one random value per key pair: repeat with 10 different seeds.

In [ ]:
seed_runs = [summarize(run_experiment(N_QUBITS, N_KEYS, seed=s)) for s in range(10)]
pw = np.array([r["p_wrong"] for r in seed_runs])
hw = np.array([r["H_wrong"] for r in seed_runs])
print(f"P_wrong: {pw.mean():.4f} +- {pw.std(ddof=1):.4f}   (1/d = {2.0**-N_QUBITS:.4f})")
print(f"H_wrong: {hw.mean():.2f} +- {hw.std(ddof=1):.2f} bits   (random state: {haar_entropy(2**N_QUBITS):.2f})")

## 8. Dependence on $n$, with computational resources

CX count and depth after transpiling to `{cx, rz, sx, x}`; wall-clock time of each simulation.

In [ ]:
def cx_and_depth(qc):
    tqc = transpile(qc, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1, seed_transpiler=SEED)
    return tqc.count_ops().get("cx", 0), tqc.depth()

def single_gate(n, gate):
    qc = QuantumCircuit(n)
    qc.append(gate, range(n))
    return qc

def write_table(path, header, rows, colspec):
    with open(path, "w") as f:
        f.write(f"\\begin{{tabular}}{{{colspec}}}\n\\toprule\n{header} \\\\\n\\midrule\n")
        for row in rows:
            f.write(" & ".join(row) + " \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n")

scan_n = []
for n in [4, 5, 6, 7, 8]:
    t0 = time.perf_counter()
    res = run_experiment(n, N_KEYS, n_messages=None if n <= 6 else 32)
    elapsed = time.perf_counter() - t0
    r = summarize(res)
    th = res["theta"][0]
    r["cx_f"], _ = cx_and_depth(single_gate(n, QFTGate(n)))
    r["cx_d"], _ = cx_and_depth(single_gate(n, DiagonalGate(list(np.exp(1j * th)))))
    _, r["depth"] = cx_and_depth(lock_circuit(n, th))
    r["time"] = elapsed
    scan_n.append(r)
    print(f"n={n}: CX(QFT)={r['cx_f']:3d}  CX(D_k)={r['cx_d']:3d}  depth={r['depth']:3d}  "
          f"P_wrong={r['p_wrong']:.4f} (1/d={2.0**-n:.4f})  H_wrong={r['H_wrong']:.2f}  time={elapsed:.1f}s")

write_table("table_n.tex",
            r"$n$ & $d$ & CX (QFT) & CX ($D_k$) & depth & $P_{\rm wrong}$ & $1/d$ & $H_{\rm wrong}$ & time [s]",
            [[str(r["n"]), str(r["d"]), str(r["cx_f"]), str(r["cx_d"]), str(r["depth"]),
              f"{r['p_wrong']:.4f}", f"{1/r['d']:.4f}", f"{r['H_wrong']:.2f}", f"{r['time']:.1f}"] for r in scan_n],
            "ccccccccc")

## 9. Dependence on $K$, and Eve's information ($n = 6$, all messages)

In [ ]:
scan_K = []
for Kx in [2, 4, 8, 16]:
    scan_K.append(summarize(run_experiment(N_QUBITS, Kx)))
    r = scan_K[-1]
    print(f"K={Kx:2d}: P_wrong={r['p_wrong']:.4f}  H_wrong={r['H_wrong']:.2f}  "
          f"I_guess={r['I_guess']:.2f}  chi={r['chi']:.2f}  (n-log2K={N_QUBITS - np.log2(Kx):.0f})")

write_table("table_K.tex",
            r"$K$ & key bits & $P_{\rm wrong}$ & $H_{\rm wrong}$ & $I_{\rm guess}$ (lower) & $\chi$ (upper) & $n-\log_2K$",
            [[str(r["K"]), f"{np.log2(r['K']):.0f}", f"{r['p_wrong']:.4f}", f"{r['H_wrong']:.2f}",
              f"{r['I_guess']:.2f}", f"{r['chi']:.2f}", f"{N_QUBITS - np.log2(r['K']):.0f}"] for r in scan_K],
            "ccccccc")

## 10. Export numbers for the report

In [ ]:
def sci(x):
    mant, exp = f"{max(float(x), 1e-17):.1e}".split("e")
    return f"${mant}\\times 10^{{{int(exp)}}}$"

r6 = next(r for r in scan_n if r["n"] == N_QUBITS)
macros = {
    "PCorrect": f"{S['p_correct']:.4f}", "HCorrect": f"{S['H_correct']:.2f}",
    "CorrectDev": sci(S["dev_correct"]),
    "PWrong": f"{S['p_wrong']:.4f}", "HWrong": f"{S['H_wrong']:.2f}",
    "PWrongMin": f"{S['p_wrong_pairs'].min():.4f}", "PWrongMax": f"{S['p_wrong_pairs'].max():.4f}",
    "PNoCorr": f"{S['p_nocorr']:.4f}", "HNoCorr": f"{S['H_nocorr']:.2f}",
    "PDirect": f"{S['p_direct']:.4f}", "HDirect": f"{S['H_direct']:.2f}",
    "HHaar": f"{haar_entropy(S['d']):.2f}",
    "IGuess": f"{S['I_guess']:.2f}", "INoCorr": f"{S['I_nocorr']:.2f}",
    "IDirect": f"{abs(S['I_direct']):.2f}", "Holevo": f"{S['chi']:.2f}",
    "ShiftDev": sci(shift_dev),
    "PWrongBin": f"{S_bin['p_wrong']:.4f}", "HWrongBin": f"{S_bin['H_wrong']:.2f}",
    "WEDelta": f"{we_delta:.2f}", "WEPcirc": f"{we_pcirc:.4f}", "WEPformula": f"{we_pform:.4f}",
    "PWrongSeeds": f"{pw.mean():.4f}", "PWrongSeedsStd": f"{pw.std(ddof=1):.4f}",
    "HWrongSeeds": f"{hw.mean():.2f}", "HWrongSeedsStd": f"{hw.std(ddof=1):.2f}",
    "CXQFT": str(r6["cx_f"]), "CXDiag": str(r6["cx_d"]), "DepthLock": str(r6["depth"]),
    "TimeMain": f"{r6['time']:.1f}",
}
with open("results.tex", "w") as f:
    for name, value in macros.items():
        f.write(f"\\newcommand{{\\{name}}}{{{value}}}\n")
print(open("results.tex").read())